In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date

In [2]:
from pyspark.sql.functions import current_timestamp, lit, expr, to_date, when, lower, upper, trim, from_json, explode, col, to_timestamp

In [3]:
spark = get_sparkSession("Load silver.fact_customer_events")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
_schema_name = "silver"
_table_name = "fact_customer_events"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "silver_fact_customer_events.ipynb"
_bronze_table = "bronze.fact_customer_events"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for silver.fact_customer_events


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)
spark.conf.set("spark.sql.parquet.mergeSchema", True)


In [8]:
#Read get_max_loadTimestamp

max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)


In [9]:
#Reading from bronze table and resolving schema

df = spark.read.table(_bronze_table) \
          .where(f"created_on > to_timestamp('{max_timestamp}')")

print(f"SPARK-APP: Bronze Table Data count : {df.count()}")

schema_df = spark.read.json(df.rdd.map(lambda row : row.value))
json_schema = schema_df.schema

schema_df.show()
print("SPARK-APP: Schema : " + str(json_schema))

SPARK-APP: Bronze Table Data count : 4
+-------+-----------+--------------------+--------------------+--------+--------------------+----------------+-------------------+------+
|channel|customer_id|              device|       event_details|event_id|     event_timestamp|      event_type|           location| store|
+-------+-----------+--------------------+--------------------+--------+--------------------+----------------+-------------------+------+
|    MKT|    CUST001|{5.2.1, null, and...|{null, [{Electron...| EVT0001|2025-01-10T10:15:30Z|    product_view|  {Seattle, US, WA}|AMZ_US|
|    MKT|    CUST002|{5.2.1, null, and...|{null, [{Electron...| EVT0002|2025-01-10T10:18:10Z|     add_to_cart|  {Seattle, US, WA}|AMZ_US|
|    DTC|    CUST003|{4.9.0, null, ios...|{null, [{Electron...| EVT0003|2025-01-12T14:22:05Z|          search|   {Mumbai, IN, MH}|   WEB|
|    MKT|    CUST004|{5.0.0, safari, m...|{credit_card, [{E...| EVT0004|2025-01-13T11:30:00Z|purchase_attempt|{Bangalore, IN, KA}|  E

In [10]:
#Attaching schema to df

df_withSchema = df.withColumn("event_data", from_json("value", json_schema)).drop("value")

df_withSchema.show(truncate = False)

+--------------------------+------------------------------+--------------------------+------------------------------+------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|created_on                |created_by                    |modified_on               |modified_by                   |source_file             |event_data                                                                                                                                                                                    |
+--------------------------+------------------------------+--------------------------+------------------------------+------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
#Explode all values

df_withSchema = df_withSchema.withColumn("products_viewed", explode(col("event_data.event_details.products_viewed"))) \
                             .select("*", "products_viewed.product_id", expr("products_viewed.category as product_category")) \
                             .drop("products_viewed")

df_withSchema.show(truncate=False)

+--------------------------+------------------------------+--------------------------+------------------------------+------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------------+
|created_on                |created_by                    |modified_on               |modified_by                   |source_file             |event_data                                                                                                                                                                                    |product_id|product_category|
+--------------------------+------------------------------+--------------------------+------------------------------+------------------------+--------------------------------------------------------------------------------------------------------------------------------------

In [12]:
#Flatten dataset

df_withSchema = df_withSchema.withColumn("event_id", col("event_data.event_id")) \
                             .withColumn("customer_id", col("event_data.customer_id")) \
                             .withColumn("channel", col("event_data.channel")) \
                             .withColumn("store", col("event_data.store")) \
                             .withColumn("event_type", col("event_data.event_type")) \
                             .withColumn("event_timestamp", col("event_data.event_timestamp")) \
                             .withColumn("city", col("event_data.location.city")) \
                             .withColumn("state", col("event_data.location.state")) \
                             .withColumn("country", col("event_data.location.country")) \
                             .withColumn("device_os", col("event_data.device.os")) \
                             .withColumn("device_type", col("event_data.device.type")) \
                             .withColumn("session_id", col("event_data.event_details.session_id")) \
                             .withColumn("payment_method", col("event_data.event_details.payment_method")) \
                             .drop("event_data")
                             

df_withSchema.show(truncate=False)

+--------------------------+------------------------------+--------------------------+------------------------------+------------------------+----------+----------------+--------+-----------+-------+------+----------------+--------------------+---------+-----+-------+---------+-----------+----------+--------------+
|created_on                |created_by                    |modified_on               |modified_by                   |source_file             |product_id|product_category|event_id|customer_id|channel|store |event_type      |event_timestamp     |city     |state|country|device_os|device_type|session_id|payment_method|
+--------------------------+------------------------------+--------------------------+------------------------------+------------------------+----------+----------------+--------+-----------+-------+------+----------------+--------------------+---------+-----+-------+---------+-----------+----------+--------------+
|2026-01-29 00:25:11.885275|raw_fact_customer_eve

In [13]:
df_dedup = df_withSchema.withColumn("_rn", expr("row_number() over (partition by event_id, product_id order by created_on desc)")) \
                        .where("_rn = 1") \
                        .drop("_rn")

loaded_count = df_dedup.count()

print(f"SPARK-APP: Table count after de-dupe : {loaded_count}")

SPARK-APP: Table count after de-dupe : 7


In [14]:
df_withSchema.show(10)
df_withSchema.printSchema()


+--------------------+--------------------+--------------------+--------------------+--------------------+----------+----------------+--------+-----------+-------+------+----------------+--------------------+---------+-----+-------+---------+-----------+----------+--------------+
|          created_on|          created_by|         modified_on|         modified_by|         source_file|product_id|product_category|event_id|customer_id|channel| store|      event_type|     event_timestamp|     city|state|country|device_os|device_type|session_id|payment_method|
+--------------------+--------------------+--------------------+--------------------+--------------------+----------+----------------+--------+-----------+-------+------+----------------+--------------------+---------+-----+-------+---------+-----------+----------+--------------+
|2026-01-29 00:25:...|raw_fact_customer...|2026-01-29 00:25:...|raw_fact_customer...|dim_customer_even...|     P1001|     Electronics| EVT0001|    CUST001|  

In [15]:
#Formating the data

df_ld = df_dedup.withColumn("event_timestamp", to_timestamp("event_timestamp"))
                

In [16]:
#Standardizing the codes and types

df_ld = df_ld.withColumn("event_id", upper(trim("event_id"))) \
             .withColumn("customer_id", upper(trim("customer_id"))) \
             .withColumn("store", upper(trim("store"))) \
             .withColumn("channel", upper(trim("channel"))) \
             .withColumn("product_id", upper(trim("product_id"))) \
             .withColumn("state", upper(trim("state"))) \
             .withColumn("country", upper(trim("country"))) \
             .withColumn("session_id", upper(trim("session_id"))) \
             .withColumn("payment_method", upper(trim("payment_method"))) \
             .withColumn("device_os", upper(trim("device_os"))) \
             .withColumn("device_type", upper(trim("device_type")))

In [17]:
#Adding audit columns

df_ld = df_ld.withColumn("created_on", current_timestamp()) \
             .withColumn("created_by", lit(_script_name)) \
             .withColumn("modified_on", current_timestamp()) \
             .withColumn("modified_by", lit(_script_name))

In [18]:
df_ld.show(10)
df_ld.printSchema()

+--------------------+--------------------+--------------------+--------------------+--------------------+----------+----------------+--------+-----------+-------+------+----------------+-------------------+---------+-----+-------+---------+-----------+----------+--------------+
|          created_on|          created_by|         modified_on|         modified_by|         source_file|product_id|product_category|event_id|customer_id|channel| store|      event_type|    event_timestamp|     city|state|country|device_os|device_type|session_id|payment_method|
+--------------------+--------------------+--------------------+--------------------+--------------------+----------+----------------+--------+-----------+-------+------+----------------+-------------------+---------+-----+-------+---------+-----------+----------+--------------+
|2026-01-29 01:02:...|silver_fact_custo...|2026-01-29 01:02:...|silver_fact_custo...|dim_customer_even...|     P1001|     Electronics| EVT0001|    CUST001|    M

In [19]:
#Writing to Table and log data to load_controller

_data = []

df_ld.write \
     .format("delta") \
     .mode("overwrite") \
     .saveAsTable(_fullname)
    
_data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to silver.fact_customer_events and load_controller


In [20]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+--------------------+--------------------+--------------------+--------------------+--------------------+----------+----------------+--------+-----------+-------+------+----------------+-------------------+---------+-----+-------+---------+-----------+----------+--------------+
|          created_on|          created_by|         modified_on|         modified_by|         source_file|product_id|product_category|event_id|customer_id|channel| store|      event_type|    event_timestamp|     city|state|country|device_os|device_type|session_id|payment_method|
+--------------------+--------------------+--------------------+--------------------+--------------------+----------+----------------+--------+-----------+-------+------+----------------+-------------------+---------+-----+-------+---------+-----------+----------+--------------+
|2026-01-29 01:02:...|silver_fact_custo...|2026-01-29 01:02:...|silver_fact_custo...|dim_customer_even...|     P1001|     Electronics| EVT0001|    CUST001|    M

In [21]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+------+-----------+--------------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| layer|schema_name|          table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+------+-----------+--------------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|silver|     silver|fact_customer_events|full-load| overwrite|2026-01-29 01:02:...|    success|           7|2026-01-29 01:02:...|silver_fact_custo...|2026-01-29 01:02:...|silver_fact_custo...|
+------+-----------+--------------------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [22]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|0      |7            |
+-------+-------------+



In [23]:
spark.stop()